### Samplig + processing patients (based on data fullness)

In [27]:
# import pandas as pd

# prediction_results = pd.read_csv('../training_ml/artifacts/prediction_results_by_hadm.csv')
# total_data_full = pd.read_parquet('../training_ml/artifacts/ind_hosp.parquet')
# correct_predictions = prediction_results[prediction_results['correct_prediction'] == True]
# test_pairs = correct_predictions[['subject_id', 'hadm_id']].drop_duplicates()
# test_pairs_tuple = list(zip(test_pairs['subject_id'], test_pairs['hadm_id']))

# total_data = total_data_full[
#     total_data_full[['subject_id', 'hadm_id']].apply(tuple, axis=1).isin(test_pairs_tuple)
# ].copy()
# total_data = total_data.merge(
#     prediction_results[['subject_id', 'hadm_id', 'predicted_proba']], 
#     on=['subject_id', 'hadm_id'], 
#     how='left'
# )

# print(f"Data filtered to test hospitalizations only:")
# print(f"  Total rows: {len(total_data)}")
# print(f"  Unique patients: {total_data['subject_id'].nunique()}")
# print(f"  Unique hospitalizations: {total_data['hadm_id'].nunique()}")

In [28]:
import pandas as pd
import numpy as np
test_df = pd.read_parquet('../training_ml/artifacts/test_data_with_predictions.parquet')
hospital_df = (
    test_df
    .sort_values(["subject_id", "hadm_id", "day"])
    .groupby(["subject_id", "hadm_id"])
    .last()
    .reset_index()
)

threshold = 0.2158
hospital_df["risk_scaled"] = np.where(
    hospital_df["predicted_proba"] <= threshold,
    0.5 * hospital_df["predicted_proba"] / threshold,
    0.5 + 0.5 * (
        (hospital_df["predicted_proba"] - threshold)
        / (hospital_df["predicted_proba"].max() - threshold)
    )
)

def prediction_group(row):
    if row.true_class == 1 and row.predicted_class == 1:
        return "TP"
    elif row.true_class == 0 and row.predicted_class == 0:
        return "TN"
    elif row.true_class == 0 and row.predicted_class == 1:
        return "FP"
    else:
        return "FN"

hospital_df["group"] = hospital_df.apply(prediction_group, axis=1)
icd_cols = [c for c in hospital_df.columns if c.startswith("icd_")]
ccsr_cols = [c for c in hospital_df.columns if c.startswith("ccsr_")]
lab_cols = [c for c in hospital_df.columns if c.startswith("lab_")]

hospital_df["has_icd"] = hospital_df[icd_cols].sum(axis=1) > 0
hospital_df["has_ccsr"] = hospital_df[ccsr_cols].sum(axis=1) > 0
hospital_df["has_labs"] = hospital_df[lab_cols].notna().any(axis=1)
hospital_df["context_type"] = np.select(
    [
        hospital_df["has_icd"] & hospital_df["has_ccsr"] & hospital_df["has_labs"],

        (hospital_df["has_icd"] | hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        hospital_df["has_labs"],

        (~hospital_df["has_icd"]) &
        (~hospital_df["has_ccsr"]) &
        (~hospital_df["has_labs"]),
    ],
    [
        "full",
        "diagnoses_only",
        "labs_only",
        "empty",
    ],
    default="other"
)
total_data = hospital_df

# Fullness-based sort
patient_info = total_data.copy()

icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

has_icd = patient_info[icd_cols].sum(axis=1) > 0
has_ccsr = patient_info[ccsr_cols].sum(axis=1) > 0
has_labs = patient_info[lab_cols].notna().any(axis=1)

patient_info["context_type"] = np.select(
    [
        has_icd & has_ccsr & has_labs,
        (has_icd | has_ccsr) & (~has_labs),
        (~has_icd) & (~has_ccsr) & has_labs,
        (~has_icd) & (~has_ccsr) & (~has_labs)
    ],
    [
        "full",
        "diagnoses_only",
        "labs_only",
        "empty"
    ],
    default="other"
)

selected = []

contexts = [
    "full",
    "diagnoses_only",
    "labs_only",
    "empty"
]

for group in ["TP", "TN", "FP", "FN"]:
    df_group = patient_info[patient_info["group"] == group]
    for context in contexts:
        df = df_group[df_group["context_type"] == context]
        if len(df) == 0:
            continue
        low = df.nsmallest(21, "predicted_proba")
        high = df.nlargest(21, "predicted_proba")
        mid = (
            df.iloc[
                (
                    df["predicted_proba"]
                    - df["predicted_proba"].median()
                ).abs().argsort()
            ]
            .head(21)
        )

        selected.extend([low, mid, high])

sample_patients_df = (
    pd.concat(selected)
      .drop_duplicates(subset="hadm_id")
      .reset_index(drop=True)
)
sample_patients_df

,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,predicted_proba,predicted_class,true_class,correct_prediction,risk_scaled,group,has_icd,has_ccsr,has_labs,context_type
0,14849966,26616701,2141-11-18,19,78,19,2141-11-17,570,247,1,...,0.215832,1,1,True,0.500032,TP,True,True,True,full
1,10184327,21418533,2131-11-18,11,82,11,2131-11-16,242,110,0,...,0.215882,1,1,True,0.500082,TP,True,True,True,full
2,17154976,23809945,2112-04-30,7,24,7,2112-04-28,77,28,0,...,0.215951,1,1,True,0.500151,TP,True,True,True,full
3,19276413,21514252,2127-06-04,4,88,4,2127-06-03,76,36,1,...,0.215956,1,1,True,0.500156,TP,True,True,True,full
4,17940493,28973861,2143-11-28,9,60,9,2143-11-26,162,99,0,...,0.216089,1,1,True,0.500289,TP,True,True,True,full
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,18286792,27013846,2129-01-21,1,32,1,2129-01-19,10,5,0,...,0.192286,0,1,False,0.445519,FN,False,False,False,empty
1004,14198510,29902595,2159-01-29,1,46,1,2159-01-28,1,0,0,...,0.191155,0,1,False,0.442898,FN,False,False,False,empty
1005,11014543,23512663,2187-04-24,1,51,1,2187-04-23,2,1,0,...,0.190881,0,1,False,0.442264,FN,False,False,False,empty
1006,10301071,29367697,2174-05-24,7,29,7,2174-05-22,14,0,0,...,0.190827,0,1,False,0.442139,FN,False,False,False,empty


In [29]:
print(len(sample_patients_df))
print(pd.crosstab(sample_patients_df['group'], sample_patients_df['context_type']))

print("Gender distribution")
print(sample_patients_df['gender_male'].value_counts())
print(sample_patients_df['gender_male'].value_counts(normalize=True) * 100)

print("\nAge and los distribution")
print(sample_patients_df[['age', 'los']].describe().loc[['mean', '50%', 'min', 'max']])

print("\nTP/TN based distribution")
print(sample_patients_df.groupby('group')[['age', 'los', 'gender_male']].mean())

icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

counts_df = total_data[['hadm_id']].copy()
counts_df['icd_count'] = total_data[icd_cols].sum(axis=1)
counts_df['ccsr_count'] = total_data[ccsr_cols].sum(axis=1)
counts_df['lab_count'] = total_data[lab_cols].notna().sum(axis=1)

counts_df = counts_df.drop_duplicates(subset=['hadm_id'])

sample_patients_df = sample_patients_df.drop(columns=['icd_count', 'ccsr_count', 'lab_count'], errors='ignore')
sample_patients_df = sample_patients_df.merge(counts_df, on='hadm_id', how='left')

print("Gender distribution")
print(sample_patients_df['gender_male'].value_counts())
print(sample_patients_df['gender_male'].value_counts(normalize=True) * 100)

print("\nAge and los distribution")
print(sample_patients_df[['age', 'los']].describe().loc[['mean', '50%', 'min', 'max']])

print("\nMedical data volume distribution (ICD, CCSR, Labs counts)")
medical_cols = ['icd_count', 'ccsr_count', 'lab_count']
print(sample_patients_df[medical_cols].describe().loc[['mean', '50%', 'min', 'max']])

print("\nTP/TN based distribution")
print(sample_patients_df.groupby('group')[['age', 'los', 'gender_male', 'icd_count', 'ccsr_count', 'lab_count']].mean())


1008
context_type  diagnoses_only  empty  full  labs_only
group                                               
FN                        63     63    63         63
FP                        63     63    63         63
TN                        63     63    63         63
TP                        63     63    63         63
Gender distribution
gender_male
0    524
1    484
Name: count, dtype: int64
gender_male
0    51.984127
1    48.015873
Name: proportion, dtype: float64

Age and los distribution
            age        los
mean  49.874008   3.456349
50%   47.500000   2.000000
min   18.000000   1.000000
max   95.000000  24.000000

TP/TN based distribution
             age       los  gender_male
group                                  
FN     49.047619  3.091270     0.396825
FP     49.571429  4.027778     0.559524
TN     51.555556  2.944444     0.404762
TP     49.321429  3.761905     0.559524
Gender distribution
gender_male
0    524
1    484
Name: count, dtype: int64
gender_male
0    51.984

In [30]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [31]:
# Age/gender-based sort
# patient_info = total_data.copy()
# icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
# ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
# lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

# patient_info['icd_count'] = patient_info[icd_cols].sum(axis=1)
# patient_info['ccsr_count'] = patient_info[ccsr_cols].sum(axis=1)
# patient_info['lab_count'] = patient_info[lab_cols].notna().sum(axis=1)

# patient_info = patient_info[
#     (patient_info['icd_count'] >= 1) & (patient_info['ccsr_count'] >= 1) &
#     (patient_info['lab_count'] >= 1)
# ].copy()

# age_groups = {
#     'young': (18, 40),
#     'middle': (40, 65),
#     'elderly': (65, 80),
#     'very_elderly': (80, 120)
# }

# def get_age_group(age):
#     for group, (low, high) in age_groups.items():
#         if low <= age < high:
#             return group
#     return 'unknown'

# patient_info = patient_info[['subject_id', 'hadm_id', 'gender_male', 'age', 'los', 'icd_count', 'ccsr_count', 'lab_count', 'target_readmission_30d', 'risk_scaled']]
# patient_info['age_group'] = patient_info['age'].apply(get_age_group)

# print(f"Patient info created:")
# print(f"  Total hospitalizations: {len(patient_info)}")
# print(f"  With readmission: {patient_info['target_readmission_30d'].sum():.0f}")
# print(f"  Without readmission: {len(patient_info) - patient_info['target_readmission_30d'].sum():.0f}")

# def get_stratified_sample(patient_info, sample_size=20, random_state=42):
#     readmission_yes = patient_info[patient_info['target_readmission_30d'] == 1]
#     readmission_no = patient_info[patient_info['target_readmission_30d'] == 0]
    
#     n_total = sample_size
#     n_yes = n_total // 2
#     n_no = n_total - n_yes
#     samples = []

#     for group, label, n in [(readmission_yes, 'with_readmission', n_yes), 
#                             (readmission_no, 'without_readmission', n_no)]:
       
#         age_groups_list = group['age_group'].unique()
#         n_per_age = n // len(age_groups_list) if len(age_groups_list) > 0 else 0
#         remainder = n - n_per_age * len(age_groups_list)
        
#         for i, age_group in enumerate(age_groups_list):
#             age_group_patients = group[group['age_group'] == age_group]

#             n_age = n_per_age + (1 if i < remainder else 0)
#             if n_age == 0:
#                 continue

#             male = age_group_patients[age_group_patients['gender_male'] == 1]
#             female = age_group_patients[age_group_patients['gender_male'] == 0]
            
#             n_male_age = min(n_age // 2, len(male))
#             n_female_age = n_age - n_male_age
            
#             if len(male) < n_male_age:
#                 n_male_age = len(male)
#                 n_female_age = n_age - n_male_age
            
#             if len(female) < n_female_age:
#                 n_female_age = len(female)
#                 n_male_age = n_age - n_female_age
            
#             if len(male) > 0 and n_male_age > 0:
#                 sample_male = male.sample(n=min(n_male_age, len(male)), random_state=random_state)
#                 samples.append(sample_male)
            
#             if len(female) > 0 and n_female_age > 0:
#                 sample_female = female.sample(n=min(n_female_age, len(female)), random_state=random_state)
#                 samples.append(sample_female)

#     if len(samples) > 0:
#         sample = pd.concat(samples).sample(frac=1, random_state=random_state)
#     else:
#         print("No samples selected")
#         return pd.DataFrame()
    
#     print(f"\nSelected {len(sample)} hospitalizations")
#     print(f"   With readmission: {sample['target_readmission_30d'].sum():.0f}")
#     print(f"   Without readmission: {len(sample) - sample['target_readmission_30d'].sum():.0f}")
#     print(f"   Male: {sample['gender_male'].sum():.0f}")
#     print(f"   Female: {len(sample) - sample['gender_male'].sum():.0f}")
    
#     print(f"\nAge groups in sample:")
#     print(sample['age_group'].value_counts().sort_index())
    
#     return sample

# SAMPLE_SIZE = 32
# sample_patients_df = get_stratified_sample(patient_info, sample_size=SAMPLE_SIZE, random_state=13)
# sample_patients_df

In [32]:
def get_patient_full_data(subject_id, hadm_id, total_data, risk_score=False):
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) &
        (total_data['hadm_id'] == hadm_id)
    ].sort_values("day")

    patient = patient_raw.iloc[-1].to_dict()

    json_context = {
        "demographics": {},
        "diagnoses": {},
        "laboratory_values": {},
        "clinical_indicators": {}
    }
    if risk_score:
        json_context = {
            "risk_score": {},
            #"readmission": {},
            "demographics": {},
            "diagnoses": {},
            "laboratory_values": {},
            "clinical_indicators": {}
        }
        
        json_context["risk_score"] = round(patient.get("risk_scaled", None), 2)
        #json_context["readmission"] = "True" if patient.get("true_class", None) == 1 else "False"

    json_context["demographics"]["gender"] = (
        "Male" if patient["gender_male"] == 1 else "Female"
    )
    json_context["demographics"]["age"] = (patient["age"])
    json_context["demographics"]["length_of_stay"] = (patient["los"])

    race_cols = [c for c in total_data.columns if c.startswith("race_")]

    for col in race_cols:
        if patient.get(col) == 1:
            json_context["demographics"]["race"] = (
                col.replace("race_", "")
                .replace("_", " ")
                .title()
            )
            break

    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["insurance"] = col.replace('insurance_', '')
            break
    if 'insurance' not in json_context["demographics"]:
        json_context["demographics"]["insurance"] = 'Unknown'

    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_type"] = col.replace('admission_type_', '').replace('_', ' ').title()
            break
    if 'admission_type' not in json_context["demographics"]:
        json_context["demographics"]["admission_type"] = 'Unknown'

    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["discharge_location"] = col.replace('discharge_location_', '').replace('_', ' ').title()
            break
    if 'discharge_location' not in json_context["demographics"]:
        json_context["demographics"]["discharge_location"] = 'Unknown'

    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            json_context["demographics"]["admission_location"] = col.replace('admission_location_', '').replace('_', ' ').title()
            break
    if 'admission_location' not in json_context["demographics"]:
        json_context["demographics"]["admission_location"] = 'Unknown'

    icd_list = []
    icd_cols = [c for c in total_data.columns if c.startswith("icd_")]
    for col in icd_cols:
        if patient.get(col) == 1:
            icd_list.append(map_feature_name(col))
    json_context["diagnoses"]["icd"] = icd_list

    ccsr_list = []
    ccsr_cols = [c for c in total_data.columns if c.startswith("ccsr_")]
    for col in ccsr_cols:
        if patient.get(col) == 1:
            ccsr_list.append(map_feature_name(col))
    json_context["diagnoses"]["ccsr"] = ccsr_list

    lab_cols = [
        c for c in total_data.columns
        if c.startswith("lab_") and c.endswith("_daily")]
    for col in lab_cols:
        value = patient.get(col)
        if pd.notna(value):
            json_context["laboratory_values"][map_feature_name(col)] = round(float(value), 2)

    other_features = [
        "num_diagnoses",
        "num_chronic",
        "comorbidity_score",
        "num_medications_daily",
        "prior_admissions_12mo",
        "cumulative_procedures",
        "cumulative_medications",
        "num_procedures_daily"
    ]

    for feat in other_features:
        value = patient.get(feat)
        if pd.notna(value):
            json_context["clinical_indicators"][feat] = float(value)

    return {
        "subject_id": int(subject_id),
        "hadm_id": int(hadm_id),
        "json_context": json_context
    }

In [33]:
import json
import numpy as np
import pandas as pd

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, pd.Series):
        return obj.to_dict()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict('records')
    elif pd.isna(obj):
        return None
    else:
        return obj

In [34]:
import random
import copy

def convert_to_unstructured_text(patient_record):
    demographics = patient_record['demographics']
    diagnoses = patient_record['diagnoses']
    labs = patient_record['laboratory_values']
    clinical = patient_record['clinical_indicators']
    
    gender = demographics['gender'].lower()
    pronoun = 'He' if gender == 'male' else 'She'
    
    text = f"The patient is a {demographics['age']}-year-old {gender}. "
    text += f"{pronoun} is of {demographics['race']} race and has {demographics['insurance']} insurance. "
    text += f"The length of hospital stay is {demographics['length_of_stay']} days. "
    
    text += f"{pronoun} was admitted through {demographics['admission_location']} with admission type {demographics['admission_type']}. "
    text += f"The discharge location is {demographics['discharge_location']}. "
    
    if diagnoses['icd']:
        text += f"The patient has the following ICD diagnoses: "
        text += ", ".join(diagnoses['icd'])
        text += ". "
    else:
        text += "No ICD diagnoses are recorded. "
    
    if diagnoses['ccsr']:
        text += f"The CCSR categories include: "
        text += ", ".join(diagnoses['ccsr'])
        text += ". "
    else:
        text += "No CCSR categories are recorded. "
    
    if labs:
        text += f"Laboratory values show: "
        lab_items = []
        for lab, value in labs.items():
            lab_items.append(f"{lab} of {value:.2f}")
        text += ", ".join(lab_items)
        text += ". "
    else:
        text += "No laboratory values are available. "
    
    if any(v is not None for v in clinical.values()):
        text += f"Clinical indicators: "
        clinical_items = []
        for key, value in clinical.items():
            if value is not None:
                clinical_items.append(f"{key} is {value:.2f}")
        text += ", ".join(clinical_items)
        text += ". "
    
    return text

def convert_to_empty_card(patient_record):
    demographics = patient_record['demographics']
    
    empty_card = {
        'demographics': {
            'age': demographics.get('age'),
            'gender': demographics.get('gender'),
            'race': demographics.get('race', 'Unknown'),
            'insurance': demographics.get('insurance', 'Unknown'),
            'admission_type': demographics.get('admission_type', 'Unknown'),
            'admission_location': demographics.get('admission_location', 'Unknown'),
        }
    }
    
    return empty_card

def convert_to_incomplete_card(patient_clean, missing_fraction=0.3, random_state=42):
    random.seed(random_state)
    incomplete = copy.deepcopy(patient_clean)
    incomplete.pop('risk_score')
    removable = []

    for lab in incomplete["laboratory_values"]:
        removable.append(("laboratory_values", lab))
    
    for feat in incomplete["clinical_indicators"]:
        removable.append(("clinical_indicators", feat))

    for dx in incomplete["diagnoses"]["icd"]:
        removable.append(("icd", dx))
    
    for dx in incomplete["diagnoses"]["ccsr"]:
        removable.append(("ccsr", dx))
    
    n_remove = max(1, int(len(removable) * missing_fraction))
    remove = random.sample(removable, n_remove)

    for category, value in remove:
        if category == "laboratory_values":
            incomplete["laboratory_values"][value] = None

        elif category == "clinical_indicators":
            incomplete["clinical_indicators"][value] = None

        elif category == "icd":
            incomplete["diagnoses"]["icd"].remove(value)

        elif category == "ccsr":
            incomplete["diagnoses"]["ccsr"].remove(value)
    return incomplete
   
def convert_to_row_column_format(patient_record):
    rows = []
    rows.extend([
        {
            "feature_category": "demographics",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["demographics"].items()
    ])

    rows.extend([
        {
            "feature_category": "diagnosis",
            "feature_name": f"ICD",
            "value": diagnosis
        }
        for i, diagnosis in enumerate(patient_record["diagnoses"]["icd"], start=1)
    ])

    rows.extend([
        {
            "feature_category": "diagnosis",
            "feature_name": f"CCSR",
            "value": diagnosis
        }
        for i, diagnosis in enumerate(patient_record["diagnoses"]["ccsr"], start=1)
    ])

    rows.extend([
        {
            "feature_category": "laboratory",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["laboratory_values"].items()
    ])

    rows.extend([
        {
            "feature_category": "clinical indicator",
            "feature_name": feature,
            "value": value
        }
        for feature, value in patient_record["clinical_indicators"].items()
    ])

    return pd.DataFrame(rows)

def convert_to_long_list(patient_clean, subject_id, hadm_id):
    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) & 
        (total_data['hadm_id'] == hadm_id)
    ]
    
    if len(patient_raw) == 0:
        return None
    
    patient = patient_raw.iloc[0].to_dict()
    
    all_features = []
    demographics = patient_clean['demographics']
    demographic_features = [
        ('age', demographics.get('age')),
        ('gender', 'Male' if demographics.get('gender_male', 0) == 1 else 'Female'),
        ('race', demographics.get('race', 'Unknown')),
        ('insurance', demographics.get('insurance', 'Unknown')),
        ('length_of_stay', demographics.get('length_of_stay', 0)),
        ('admission_type', demographics.get('admission_type', 'Unknown')),
        ('admission_location', demographics.get('admission_location', 'Unknown')),
        ('discharge_location', demographics.get('discharge_location', 'Unknown'))
    ]
    
    for name, value in demographic_features:
        all_features.append(f"[Demographics] {name}: {value}")
    
    icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
    for col in icd_cols:
        value = patient.get(col, 0)
        display_name = map_feature_name(col)
        value = int(value) if pd.notna(value) else value
        all_features.append(f"[ICD] {display_name}: {value}")
    
    ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
    for col in ccsr_cols:
        value = patient.get(col, 0)
        display_name = map_feature_name(col)
        all_features.append(f"[CCSR] {display_name}: {value}")
    
    lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]
    for col in lab_cols:
        value = patient.get(col)
        display_name = map_feature_name(col)
        all_features.append(f"[Laboratory] {display_name}: {value}")
    
    clinical_features = [
        'num_diagnoses', 'num_chronic', 'comorbidity_score',
        'num_medications_daily', 'prior_admissions_12mo',
        'cumulative_procedures', 'cumulative_medications',
        'num_procedures_daily'
    ]
    
    for feat in clinical_features:
        value = patient.get(feat)
        all_features.append(f"[Clinical] {feat}: {value}")
    
    return convert_to_serializable(all_features)

In [35]:
import traceback
import hashlib
def save_all_formats_to_single_json(sample_patients_df, total_data, output_file='all_patients_formats.json'):
    
    all_patients_formatted = []
    failed_patients = []
    
    print(f"Processing {len(sample_patients_df)} patients...")
    
    for idx, row in sample_patients_df.iterrows():
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        
        try:
            clean_data = get_patient_full_data(subject_id, hadm_id, total_data, True)
            clean_data = convert_to_serializable(clean_data)
            clean_data = clean_data['json_context']
            unstructured_context = convert_to_unstructured_text(clean_data)
            empty_context = convert_to_empty_card(clean_data)
            row_column_context = convert_to_row_column_format(clean_data)
            rs = int(hashlib.md5(f"{subject_id}_{hadm_id}".encode()).hexdigest(), 16) % (2**32)
            incomplete_context = convert_to_incomplete_card(clean_data, random_state=rs)
            long_list_context = convert_to_long_list(clean_data, subject_id, hadm_id)

            risk_score = clean_data['risk_score']
            clean_data.pop('risk_score')
            patient_entry = {
                'subject_id': subject_id,
                'hadm_id': hadm_id,
                'risk_score': risk_score,
                'json_context': clean_data,
                'unstructured_context': unstructured_context,
                'empty_context': empty_context,
                'row_column_context': row_column_context,
                'long_list_context': long_list_context,
                'incomplete_context': incomplete_context
            }
            
            all_patients_formatted.append(patient_entry)
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id}")
            
        except Exception as e:
            failed_patients.append(f"{subject_id}_{hadm_id}")
            traceback.print_exc()
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id} - ERROR: {e}")
    
    output_data = {
        'total_patients': len(all_patients_formatted),
        'failed_patients': failed_patients,
        'patients': all_patients_formatted
    }
    
    output_data = convert_to_serializable(output_data)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"\nSuccess: {len(all_patients_formatted)} patients saved to: {output_file}")
    if failed_patients:
        print(f"Failed: {len(failed_patients)} patients")
    
    return output_data

all_formats_data = save_all_formats_to_single_json(
    sample_patients_df,
    total_data,
    output_file='data/all_patients.json'
)

Processing 1008 patients...
Patient 1/1008: 14849966_26616701
Patient 2/1008: 10184327_21418533
Patient 3/1008: 17154976_23809945
Patient 4/1008: 19276413_21514252
Patient 5/1008: 17940493_28973861
Patient 6/1008: 12763100_24798618
Patient 7/1008: 18375223_24980184
Patient 8/1008: 15146009_21716326
Patient 9/1008: 18631629_28813536
Patient 10/1008: 19653727_25117465
Patient 11/1008: 10738077_27694495
Patient 12/1008: 16592280_21383186
Patient 13/1008: 12609810_23837769
Patient 14/1008: 19120111_26110926
Patient 15/1008: 12915209_24676152
Patient 16/1008: 19909210_23899783
Patient 17/1008: 18936629_25256393
Patient 18/1008: 11248793_26435839
Patient 19/1008: 14941666_22983290
Patient 20/1008: 19001252_24531176
Patient 21/1008: 13875316_22913699
Patient 22/1008: 11668146_22430601
Patient 23/1008: 12743733_25444935
Patient 24/1008: 17653099_28649667
Patient 25/1008: 13397773_26500717
Patient 26/1008: 19471531_25741342
Patient 27/1008: 19613765_25402372
Patient 28/1008: 12449555_22630450
P

### Getting SHAP+background values

In [36]:
import pandas as pd
deltas = pd.read_csv("../ranking_methods/data/total_deltas.csv")
features_to_analyze = deltas['feature'].unique()

shap_bck = pd.read_csv("../ranking_methods/data/shap_matrix_background.csv")
shap_abs_bck = pd.read_csv("../ranking_methods/data/shap_matrix_abs_background.csv")
shap_long_bck = shap_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean')
shap_abs_long_bck = shap_abs_bck.melt(id_vars=['hadm_id'], var_name='feature', value_name='shap_mean_abs')
shap_combined_bck = shap_long_bck.merge(shap_abs_long_bck, on=['hadm_id', 'feature'], how='inner')

shap_combined_bck = shap_combined_bck[shap_combined_bck['feature'].isin(features_to_analyze)]

In [37]:
import json
with open('data/all_patients.json', 'r', encoding='utf-8') as f:
    ehrs = json.load(f)

ehr_lookup = {
    (x["subject_id"], x["hadm_id"]): x["json_context"]
    for x in ehrs['patients']
}

clinical_ranges = {
    'lab_50983_daily': {'low': 135, 'high': 145}, #sodium
    'lab_50971_daily': {'low': 3.5, 'high': 5}, #potassium
    'lab_50902_daily': {'low': 95, 'high': 105}, #chloride
    'lab_50882_daily': {'low': 22, 'high': 32}, #bicarbonate
    'lab_50912_daily': {'low': 0.8, 'high': 1.3}, #creatinine
    'lab_51006_daily': {'low': 8, 'high': 21}, #BUN
    'lab_50931_daily': {'low': 65, 'high': 110}, #glucose
    'lab_50893_daily': {'low': 8.5, 'high': 10.2}, #calcium
    'lab_50868_daily': {'low': 10, 'high': 18}, #anion gap
    'lab_51222_daily': {
        'male': {'low': 13, 'high': 17},
        'female': {'low': 12, 'high': 15}
    }, #hemoglobin
    'lab_51301_daily': {'low': 4, 'high': 10}, #WBC
    'lab_51265_daily': {'low': 150, 'high': 400}, #Platelet Count
    'lab_51221_daily': {
        'male': {'low': 40, 'high': 52},
        'female': {'low': 36, 'high': 47}
    }, #Hematocrit
    'lab_51250_daily': {'low': 80, 'high': 100}, #MCV
    'lab_51277_daily': {'low': 11.5, 'high': 14.5}, #RDW
    'lab_50960_daily': {'low': 1.5, 'high': 2}, #magnesium
    'lab_50970_daily': {'low': 2.7, 'high': 4.5}, #phosphate
    'lab_51248_daily': {'low': 26, 'high': 32}, #MCH
    'lab_51249_daily': {'low': 30, 'high': 35}, #MCHC
    'lab_51279_daily': {
        'male': {'low': 4.5, 'high': 5.9},
        'female': {'low': 4, 'high': 5.2}
    } #RBC
}

In [38]:
def correct_shap_signs(top_features, ehr_info):
    corrected_features = {}
    
    is_male = ehr_info.get("gender_male", 0)
    gender_key = "male" if (is_male == 1 or is_male is True) else "female"
    
    patient_labs = ehr_info.get('laboratory_values', {})
    clinical_indicators = ehr_info.get('clinical_indicators', {})
    demographics = ehr_info.get('demographics', {})

    for feature, shap_val in top_features.items():
        corrected_val = shap_val
        
        always_increase_risk_features = [
            'Obesity (END009)', 
            'Heart failure (CIR019)', 
            'Hyperlipidemia, unspecified (E785)', 
            'Disorders of lipid metabolism (END010)',
            'Diabetes mellitus with complication (END003)'
        ]
        if any(x.lower() in feature.lower() for x in always_increase_risk_features):
            if shap_val < 0:
                corrected_val = abs(shap_val)

        elif 'medications' in feature.lower():
            actual_val = float(clinical_indicators.get(feature, 0))
            
            if actual_val == 0:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val >= 5:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif 'procedures' in feature.lower():
            actual_val = float(clinical_indicators.get(feature, 0))
            
            if actual_val == 0:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val > 5:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif 'age' in feature.lower():
            actual_val = float(demographics.get(feature, 0))
            if actual_val < 65:
                if shap_val > 0:
                    corrected_val = -abs(shap_val)
            elif actual_val >= 65:
                if shap_val < 0:
                    corrected_val = abs(shap_val)

        elif feature in ['comorbidity_score', 'num_chronic', 'num_diagnoses']:
            actual_indicator_str = clinical_indicators.get(feature)
            
            if actual_indicator_str is not None:
                actual_ind_val = float(actual_indicator_str)
                
                thresholds = {
                    'comorbidity_score': 2.0,
                    'num_chronic': 2.0,
                    'num_diagnoses': 7.0
                }
                
                max_safe_value = thresholds.get(feature, 0)
                
                if actual_ind_val <= max_safe_value:
                    if shap_val > 0:
                        corrected_val = -abs(shap_val)
                        
                else:
                    if shap_val < 0:
                        corrected_val = abs(shap_val)

        else:
            import re
            lab_id_match = re.search(r'\((\d{5})\)', feature)
            
            if lab_id_match:
                lab_id = lab_id_match.group(1)
                range_key = f"lab_{lab_id}_daily"
                
                if range_key in clinical_ranges:
                    actual_lab_str = patient_labs.get(feature)
                    
                    if actual_lab_str is not None:
                        actual_lab_val = float(actual_lab_str)
                        norm_data = clinical_ranges[range_key]
                        
                        if 'male' in norm_data or 'female' in norm_data:
                            low_limit = norm_data[gender_key]['low']
                            high_limit = norm_data[gender_key]['high']
                        else:
                            low_limit = norm_data['low']
                            high_limit = norm_data['high']
                        
                        if low_limit <= actual_lab_val <= high_limit:
                            if shap_val > 0:
                                corrected_val = -abs(shap_val)

        corrected_features[feature] = corrected_val

    return corrected_features

In [39]:
shap_combined_bck['feature'] = shap_combined_bck['feature'].apply(map_feature_name)
shap_fin = []
for idx, row in sample_patients_df.iterrows():
    subject_id = row['subject_id']
    hadm_id = row['hadm_id']
    shap_cur = shap_combined_bck[shap_combined_bck['hadm_id'] == hadm_id].sort_values(by='shap_mean_abs', ascending=False)
    shap_dict = shap_cur.set_index('feature')['shap_mean'].to_dict()

    ehr_info = ehr_lookup.get((subject_id, hadm_id))
    diagnoses_icd = ehr_info['diagnoses'].get('icd', [])
    diagnoses_ccsr = ehr_info['diagnoses'].get('ccsr', [])
    lab_vals = list(ehr_info.get('laboratory_values', {}).keys())
    other_features = [
        'num_diagnoses',
        'num_chronic',
        'comorbidity_score',
        'num_medications_daily',
        'prior_admissions_12mo',
        'cumulative_procedures',
        'cumulative_medications',
        'num_procedures_daily',
        'gender_male',
        'age'
    ]

    allowed_features = set(diagnoses_icd + diagnoses_ccsr + lab_vals + other_features)
    shap_top_bck = {
        feature: value
        for feature, value in shap_dict.items()
        if feature in allowed_features
    }
    top_features = dict(list(shap_top_bck.items())[:10])
    #top_features_fin = correct_shap_signs(top_features, ehr_info)
    top_features_fin = top_features
    patient_entry = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'shap_bck_values': top_features_fin,
    }
    shap_fin.append(patient_entry)
with open('data/shap_bck_all_patients.json', 'w', encoding='utf-8') as f:
    json.dump(shap_fin, f, ensure_ascii=False, indent=2)